# Data Preparation: Survival Analysis
This notebook processes the raw transactions to construct a clean survival dataset with one row per user.

### Methodology
- **Backdated Records**: Transactions with negative subscription durations are excluded.
- **Overlapping Subscriptions**: The chronological sequence of transactions is used. A new transaction may extend the existing expiry.
- **Churn Definition**: Churn occurs if a user fails to renew within **30 days** of their `membership_expire_date`.
- **Right Censoring**: If the observation window (ending 2017-03-31) concludes within 30 days of the user's final `membership_expire_date`, they are right-censored, because it is impossible to observe whether they would have renewed within the full 30-day window.
- **Endpoints**: The survival endpoint is anchored to the user's *first* observed churn event. If they never churned, their *final* censored event is used.
- **Baseline Covariates**: To avoid data leakage, subscription covariates (price, auto-renew, etc.) are extracted from the user's *first* transaction.

In [1]:
import gc
import os

import numpy as np
import pandas as pd

## 1. Load Data

In [2]:
print('Loading datasets with optimized datatypes...')
dtypes = {
    'payment_method_id': 'int8',
    'payment_plan_days': 'int16',
    'plan_list_price': 'int16',
    'actual_amount_paid': 'int16',
    'is_auto_renew': 'int8',
    'is_cancel': 'int8'
}

transactions = pd.read_csv('../data/transactions.csv', dtype=dtypes)
trans2 = pd.read_csv('../data/transactions_v2.csv', dtype=dtypes)

# Combine and immediately delete trans2 to free RAM
transactions = pd.concat([transactions, trans2], ignore_index=True)
del trans2
gc.collect()

print("Converting dates...")
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], format='%Y%m%d')
transactions['membership_expire_date'] = pd.to_datetime(transactions['membership_expire_date'], format='%Y%m%d')

initial_len = len(transactions)
transactions.drop_duplicates(inplace=True)
print(f'Dropped {initial_len - len(transactions):,} exact duplicates.')

print(f'Loaded {len(transactions):,} unique transactions')
print(f'Unique users: {transactions["msno"].nunique():,}')

Loading datasets...


Dropped 3,339 exact duplicates.
Loaded 22,975,416 unique transactions


Unique users: 2,426,143


## 2. Clean Data
Remove negative subscription lengths (backdated records).

In [3]:
# Derive subscription length
transactions['subscription_length_days'] = (
    transactions['membership_expire_date'] - transactions['transaction_date']
).dt.days

# Keep only valid forward-looking transactions (filter in-place to save memory)
transactions.drop(transactions[transactions['subscription_length_days'] < 0].index, inplace=True)

# Sort chronologically in-place
transactions.sort_values(["msno", "transaction_date"], inplace=True)

transactions_clean = transactions
print(f'Records after cleaning backdated transactions: {len(transactions_clean):,}')

Records after cleaning backdated transactions: 22,816,913


## 3. Define Churn and Censoring Logic

In [4]:
# Calculate gap to next transaction
transactions_clean['next_transaction_date'] = (
    transactions_clean.groupby('msno')['transaction_date'].shift(-1)
)

transactions_clean['gap_after_expiry'] = (
    transactions_clean['next_transaction_date'] - transactions_clean['membership_expire_date']
).dt.days

OBSERVATION_END = transactions_clean['transaction_date'].max()
CHURN_WINDOW = pd.Timedelta(days=30)

# Tag final transaction per user
transactions_clean['is_last_transaction'] = (
    transactions_clean.groupby('msno')['transaction_date'].transform('max') == transactions_clean['transaction_date']
)

# Observed churn: Next transaction exists, but gap > 30 days
observed_churn = (
    transactions_clean['next_transaction_date'].notna() & 
    (transactions_clean['gap_after_expiry'] > 30)
)

# Final transaction churn: It's the final transaction AND 30 days have passed since expiry
final_transaction_observed_long_enough = (
    transactions_clean['is_last_transaction'] & 
    transactions_clean['membership_expire_date'].notna() & 
    (transactions_clean['membership_expire_date'] + CHURN_WINDOW <= OBSERVATION_END)
)
final_transaction_churn = (
    final_transaction_observed_long_enough & 
    transactions_clean['next_transaction_date'].isna()
)

# Combine into a single churn flag
transactions_clean['is_churn_point'] = observed_churn | final_transaction_churn

# Censored: User reaches observation end before we can establish churn
transactions_clean['is_censored'] = (
    transactions_clean['is_last_transaction'] & ~transactions_clean['is_churn_point']
)

## 4. Extract Endpoints and Baseline Covariates

In [5]:
# 1. Grab the FIRST churn event for users who churned
churned_endpoints = (
    transactions_clean[transactions_clean['is_churn_point']]
    .groupby('msno', as_index=False)
    .first()
)

# 2. For users who NEVER churned, grab their censored endpoint
censored_users_mask = transactions_clean['is_censored'] & ~transactions_clean['msno'].isin(churned_endpoints['msno'])
censored_endpoints = (
    transactions_clean[censored_users_mask]
    .groupby('msno', as_index=False)
    .last()
)

# Combine endpoints
final_endpoints = pd.concat([churned_endpoints, censored_endpoints], ignore_index=True)
print(f'Total endpoint records: {len(final_endpoints):,}')

Total endpoint records: 2,417,562


In [6]:
# Extract the FIRST transaction per user for baseline covariates (avoids data leakage)
baseline_covariates = (
    transactions_clean
    .groupby('msno', as_index=False)
    .first()
)[['msno', 'transaction_date', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'is_cancel']]

# Rename transaction_date to start_date for clarity
baseline_covariates = baseline_covariates.rename(columns={'transaction_date': 'start_date'})

## 5. Build Survival Table

In [7]:
survival_table = final_endpoints[['msno', 'membership_expire_date', 'is_churn_point']].copy()

# 1 = observed churn, 0 = right-censored
survival_table['event'] = survival_table['is_churn_point'].astype(int)

# Set endpoint date
survival_table['endpoint_date'] = np.where(
    survival_table['event'] == 1,
    survival_table['membership_expire_date'],
    OBSERVATION_END
)
survival_table['endpoint_date'] = pd.to_datetime(survival_table['endpoint_date'])

# Merge with baseline covariates
survival_table = survival_table.merge(baseline_covariates, on='msno', how='left')

# Calculate total duration in days
survival_table['duration_days'] = (
    survival_table['endpoint_date'] - survival_table['start_date']
).dt.days

# Keep final columns
final_cols = [
    'msno', 'start_date', 'endpoint_date', 'duration_days', 'event',
    'payment_method_id', 'payment_plan_days', 'plan_list_price', 
    'actual_amount_paid', 'is_auto_renew', 'is_cancel'
]
survival_table = survival_table[final_cols]

print(f'Survival table shape: {survival_table.shape}')

Survival table shape: (2417562, 11)


## 6. Sanity Checks

In [8]:
# 1. Assert exactly one row per user
assert len(survival_table) == survival_table['msno'].nunique(), 'ERROR: Multiple rows per user detected!'

# 2. Assert no negative durations
assert (survival_table['duration_days'] < 0).sum() == 0, 'ERROR: Negative durations detected!'

# 3. Print Event Breakdown
print('Event Breakdown (1=Churn, 0=Censored):')
print(survival_table['event'].value_counts(normalize=True).round(3) * 100)

# 4. Check for missing values
print('\nMissing Values:')
print(survival_table.isna().sum())

print('\nAll sanity checks passed!')

Event Breakdown (1=Churn, 0=Censored):
event
1    62.5
0    37.5
Name: proportion, dtype: float64

Missing Values:


msno                  0
start_date            0
endpoint_date         0
duration_days         0
event                 0
payment_method_id     0
payment_plan_days     0
plan_list_price       0
actual_amount_paid    0
is_auto_renew         0
is_cancel             0
dtype: int64

All sanity checks passed!


## 7. Export

In [9]:
os.makedirs('../data', exist_ok=True)
survival_table.to_csv('../data/survival_table.csv', index=False)
print('Saved survival_table.csv')

Saved survival_table.csv
